In [24]:
import numpy as np
import pandas as pd
import joblib

from cesnet_datazoo.datasets import CESNET_QUIC22
from cesnet_datazoo.config import DatasetConfig, AppSelection

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

In [25]:
dataset = CESNET_QUIC22(
    data_root="/datasets/CESNET-QUIC22",
    size="XS"
)

config = DatasetConfig(
    dataset=dataset,
    apps_selection=AppSelection.ALL_KNOWN,
    train_period_name="W-2022-44",
    test_period_name="W-2022-45"
)

dataset.set_dataset_config_and_initialize(config)

train_dataframe = dataset.get_train_df()
validation_dataframe = dataset.get_val_df()
test_dataframe = dataset.get_test_df()

print(train_dataframe.shape)
print(validation_dataframe.shape)
print(test_dataframe.shape)

Loading data from dataloader


100%|█████████████████████████████████████████████████████████████████████████████| 8162/8162 [01:13<00:00, 110.75it/s]


Loading data from dataloader


100%|████████████████████████████████████████████████████████████████████████████████| 192/192 [00:31<00:00,  6.17it/s]


Loading data from dataloader


100%|██████████████████████████████████████████████████████████████████████████████| 1247/1247 [00:59<00:00, 20.99it/s]


(1566944, 13)
(391736, 13)
(2551901, 13)


In [26]:
targets = [
    "google-docs",
    "google-drive",
    "youtube"
]

target_ids = dataset.class_info.encoder.transform(targets)

print("Target IDs:", target_ids)

train_dataframe = train_dataframe[
    train_dataframe["APP"].isin(target_ids)
]

validation_dataframe = validation_dataframe[
    validation_dataframe["APP"].isin(target_ids)
]

test_dataframe = test_dataframe[
    test_dataframe["APP"].isin(target_ids)
]

print(train_dataframe["APP"].value_counts())

print(
    dataset.class_info.encoder.inverse_transform(
        train_dataframe["APP"].unique()
    )
)

Target IDs: [ 47  48 100]
APP
100    75294
47     11019
48      5911
Name: count, dtype: int64
['youtube' 'google-drive' 'google-docs']


In [27]:
TARGET = "APP"

X_train = train_dataframe.drop(columns=[TARGET])
y_train = train_dataframe[TARGET]

X_val = validation_dataframe.drop(columns=[TARGET])
y_val = validation_dataframe[TARGET]

X_test = test_dataframe.drop(columns=[TARGET])
y_test = test_dataframe[TARGET]

In [28]:
def preprocess_dataframe(dataframe, n_packets=5):

    ppi = np.stack(dataframe["PPI"].values)

    # Keep first N packets
    ppi = ppi[:, :, :n_packets]

    # Flatten
    ppi = ppi.reshape(len(dataframe), -1)

    ppi_dataframe = pd.DataFrame(
        ppi,
        columns=[
            f"PPI_{i}"
            for i in range(ppi.shape[1])
        ],
        index=dataframe.index
    )

    dataframe = dataframe.drop(columns=["PPI"])

    dataframe = pd.concat(
        [dataframe, ppi_dataframe],
        axis=1
    )

    return dataframe

In [29]:
X_train = preprocess_dataframe(X_train, n_packets=5)
X_val = preprocess_dataframe(X_val, n_packets=5)
X_test = preprocess_dataframe(X_test, n_packets=5)

X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

print(X_train.shape)
print(X_train.dtypes.value_counts())

(92224, 26)
float32    26
Name: count, dtype: int64


In [30]:
lightgbm_model = LGBMClassifier(
    objective="multiclass",
    boosting_type="gbdt",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lightgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss"
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.021300 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2966
[LightGBM] [Info] Number of data points in the train set: 92224, number of used features: 23
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


LGBMClassifier(class_weight='balanced', colsample_bytree=0.8,
               learning_rate=0.05, n_estimators=300, n_jobs=-1,
               objective='multiclass', random_state=42, subsample=0.8)

In [31]:
predictions = lightgbm_model.predict(X_test)

In [32]:
print("Accuracy:", accuracy_score(y_test, predictions))

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_test, predictions)
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        predictions,
        average="macro"
    )
)

Accuracy: 0.9103981402208488
Balanced Accuracy: 0.8201776149506937
Macro F1: 0.7866122442590306


In [34]:
cm = confusion_matrix(
    y_test,
    predictions,
    labels=target_ids
)

cm_dataframe = pd.DataFrame(
    cm,
    index=targets,
    columns=targets
)

print(cm_dataframe)

              google-docs  google-drive  youtube
google-docs         13687          1677     2612
google-drive          640          7445     1749
youtube              2993          4667   124549


In [35]:
from catboost import CatBoostClassifier

catboost_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=300,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=100
)

catboost_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val)
)

0:	learn: 1.0339414	test: 1.0338720	best: 1.0338720 (0)	total: 274ms	remaining: 1m 21s
100:	learn: 0.2656074	test: 0.2677204	best: 0.2677204 (100)	total: 7.89s	remaining: 15.5s
200:	learn: 0.2160255	test: 0.2190838	best: 0.2190838 (200)	total: 15.2s	remaining: 7.48s
299:	learn: 0.1917885	test: 0.1960505	best: 0.1960505 (299)	total: 22.2s	remaining: 0us

bestTest = 0.1960505352
bestIteration = 299



CatBoostClassifier(depth=6, eval_metric='MultiClass', iterations=300, learning_rate=0.05, loss_function='MultiClass', random_seed=42, verbose=100)

In [44]:
catboost_predictions = catboost_model.predict(X_test).flatten()

catboost_accuracy = accuracy_score(y_test, catboost_predictions)
catboost_balanced = balanced_accuracy_score(y_test, catboost_predictions)
catboost_f1 = f1_score(
    y_test,
    catboost_predictions,
    average="macro"
)

print("Accuracy:", catboost_accuracy)
print("Balanced Accuracy:", catboost_balanced)
print("Macro F1:", catboost_f1)

print(
    classification_report(
        y_test,
        predictions,
        labels=[47,48,100],
        target_names=[
            "google-docs",
            "google-drive",
            "youtube"
        ],
        digits=4,
        zero_division=0
    )
)

Accuracy: 0.9071047813072197
Balanced Accuracy: 0.654970311912673
Macro F1: 0.7263525905971683
              precision    recall  f1-score   support

 google-docs     0.7902    0.7614    0.7756     17976
google-drive     0.5399    0.7571    0.6303      9834
     youtube     0.9662    0.9421    0.9540    132209

    accuracy                         0.9104    160019
   macro avg     0.7654    0.8202    0.7866    160019
weighted avg     0.9202    0.9104    0.9140    160019



In [39]:
from sklearn.ensemble import RandomForestClassifier

randomforest_model = RandomForestClassifier(
    n_estimators=300,
    criterion="gini",
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

randomforest_model.fit(
    X_train,
    y_train
)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)

In [43]:
randomforest_predictions = randomforest_model.predict(X_test)

randomforest_accuracy = accuracy_score(
    y_test,
    randomforest_predictions
)

randomforest_balanced = balanced_accuracy_score(
    y_test,
    randomforest_predictions
)

randomforest_f1 = f1_score(
    y_test,
    randomforest_predictions,
    average="macro"
)

print("Accuracy:", randomforest_accuracy)
print("Balanced Accuracy:", randomforest_balanced)
print("Macro F1:", randomforest_f1)

print(
    classification_report(
        y_test,
        predictions,
        labels=[47,48,100],
        target_names=[
            "google-docs",
            "google-drive",
            "youtube"
        ],
        digits=4,
        zero_division=0
    )
)

Accuracy: 0.9213343415469413
Balanced Accuracy: 0.7124569948125242
Macro F1: 0.7793214600178437
              precision    recall  f1-score   support

 google-docs     0.7902    0.7614    0.7756     17976
google-drive     0.5399    0.7571    0.6303      9834
     youtube     0.9662    0.9421    0.9540    132209

    accuracy                         0.9104    160019
   macro avg     0.7654    0.8202    0.7866    160019
weighted avg     0.9202    0.9104    0.9140    160019



In [42]:
print(np.unique(y_train))
print(np.unique(y_test))
print(np.unique(predictions))

[ 47  48 100]
[ 47  48 100]
[ 47  48 100]
